In [1]:
import os
import GEOparse
import pandas as pd


# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[sample_title] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# Remove rows where index contains "repl" suffix (technical replicates)
metadata_df = metadata_df[~metadata_df.index.str.contains('repl$')]
metadata_df.head()

,age,Sex,overall survival (os) (month),outcome,t stage,n stage,ajcc/uicc-tnm stage
samples,,,,,,,
CRC_1,63,Male,34,Dead,T4b,N0,IV
CRC_2,72,Female,34,Alive,T3,N2a,IV
CRC_3,58,Male,33,Alive,T4a,N2a,IV
CRC_4,73,Male,32,Alive,T4a,N2b,IV
CRC_5,66,Female,28,Dead,T3,N2a,IV


# Use author provided expression data


In [2]:
import requests

url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE192667&format=file"
out_path = "GSE192667_RAW.tar"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

In [3]:
import tarfile
import gzip

# Extract and load compressed files
def load_geo_data(tar_path):
    dataframes_tpm = {}
    dataframes_raw_counts = {}
    with tarfile.open(tar_path, 'r') as tar:
        for member in tar.getmembers():
            # Extract file
            f = tar.extractfile(member)

            # Decompress and read
            with gzip.open(f, 'rt') as gz_file:
                df = pd.read_csv(gz_file, sep='\t', index_col=0)
                    
            # Clean filename for dict key
            key =  '_'.join(member.name.split('_')[1:3]) 
            if member.name.endswith('_TPM.txt.gz'):
                dataframes_tpm[key] = df
            elif member.name.endswith('_count_file.txt.gz'):
                # here we ignore the Gene_length_bp column. Can be included if needed.
                df = df.rename(columns={'featureCounts': key})
                df = df.drop(columns=['Gene_length_bp'])
                dataframes_raw_counts[key] = df
    
    return dataframes_tpm, dataframes_raw_counts

# Load all files
tpm_data, raw_counts_data = load_geo_data("GSE192667_RAW.tar")

In [4]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["EnsemblGeneID", "Symbol"]]
    .dropna(subset=["EnsemblGeneID", "Symbol"])
    .set_index("EnsemblGeneID")["Symbol"]
    .to_dict()
)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_3661/392905971.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


In [5]:
# tpm_data_combined = pd.concat(tpm_data.values(), axis=1)
# tpm_data_combined = tpm_data_combined.T
# tpm_data_combined.index.name = "Sample"
# tpm_data_combined.index = tpm_data_combined.index.str.replace('_TPM', '')
# tpm_data_combined.columns.name = None

raw_counts_combined = pd.concat(raw_counts_data.values(), axis=1)
raw_counts_combined = raw_counts_combined.T
raw_counts_combined.index.name = "samples"
raw_counts_combined.columns.name = None

mappable = [c for c in raw_counts_combined.columns if c in gene_mapping]
raw_counts_combined = raw_counts_combined[mappable].rename(columns=gene_mapping)

In [6]:
# data_file_name_tpm = f'{GEO_ID}_TPM.csv'
# df_tpm = metadata_df.join(tpm_data_combined, how='inner')
# df_tpm.reset_index(inplace=True)
# df_tpm.to_csv(data_file_name_tpm, index=False)

data_file_name = f'{GEO_ID}_raw_counts.csv'
df_raw_counts = metadata_df.join(raw_counts_combined, how='inner')
df_raw_counts.reset_index(inplace=True)
df_raw_counts.to_csv(data_file_name, index=False)


In [7]:
df_raw_counts

,samples,age,Sex,overall survival (os) (month),outcome,t stage,n stage,ajcc/uicc-tnm stage,MIR6859-1,MIR1302-2,...,LSS,MCM3AP-AS1,MCM3AP,YBEY,C21orf58,PCNT,DIP2A,DIP2A-IT1,S100B,PRMT2
0,CRC_1,63,Male,34,Dead,T4b,N0,IV,0,199,...,10964,1559,5540,294,130,5795,3302,0,82,2108
1,CRC_2,72,Female,34,Alive,T3,N2a,IV,0,297,...,4035,1845,6331,469,184,9127,4858,0,90,2030
2,CRC_3,58,Male,33,Alive,T4a,N2a,IV,0,376,...,4301,1586,5374,202,54,8549,3711,4,176,2167
3,CRC_4,73,Male,32,Alive,T4a,N2b,IV,1,633,...,3388,1626,5798,168,107,7314,4391,0,185,1978
4,CRC_5,66,Female,28,Dead,T3,N2a,IV,0,1402,...,4077,1361,4973,302,185,9432,4839,0,351,2754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,CRC_85,74,Male,17,Dead,T4a,N1,IV,0,10,...,4131,1115,4839,206,39,7123,2838,1,120,1551
85,CRC_86,43,Male,24,Dead,T4a,N2a,IV,1,0,...,2021,1103,4193,586,86,6176,2797,0,20,1373
86,CRC_87,80,Male,45,Alive,T4b,N2a,IIIC,0,41,...,5875,1413,5752,365,115,6002,4453,0,32,1735
87,CRC_88,52,Female,42,Dead,T4a,N0,IIB,0,6,...,2649,1947,6853,366,171,11744,4728,2,117,1453


# Generate description

In [8]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID, limit_samples=10)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.reset_index().shape[0],
    'clinical_features': metadata_df.reset_index().shape[1],
    'genes': raw_counts_combined.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f,ensure_ascii=False, indent=4)


Description updated
